In [3]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, log_loss
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.models import Model
import warnings
warnings.filterwarnings("ignore")

In [7]:
dir_path = "/content/drive/MyDrive/archive 2/train"

In [8]:
datagen = ImageDataGenerator(rescale=1.0/255.,
                             validation_split = 0.15,
                            )
train_generator = datagen.flow_from_directory(dir_path,
                    target_size=(150,150),
                    batch_size= 32,
                    class_mode='categorical',
                    subset='training',
                    shuffle=True)
test_generator = datagen.flow_from_directory(dir_path,
                    target_size=(150,150),
                    batch_size= 32,
                    class_mode='categorical',
                    subset='validation')

Found 234 images belonging to 2 classes.
Found 41 images belonging to 2 classes.


In [11]:
tf.random.set_seed(25)
model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(150,150,3)), # 3-> colourd image (red,blue, green) and look for X_test fror size
    tf.keras.layers.Conv2D(16,kernel_size=(3,3),activation='relu'),# filter = 16
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
    tf.keras.layers.Conv2D(32,kernel_size=(3,3),activation='relu'), # filter = 32
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(256, activation='relu'), # fully connected layer
    tf.keras.layers.Dense(2, activation='softmax')]) # Changed to 2 units to match 2 classes

print(model.summary())

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_96 (Conv2D)              │ (None, 148, 148, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 74, 74, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_97 (Conv2D)              │ (None, 72, 72, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │    10,617,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,622,690 (40.52 MB)

 Trainable params: 10,622,690 (40.52 MB)

 Non-trainable params: 0 (0.00 B)

None


In [12]:
monitor = EarlyStopping(monitor='val_loss',min_delta=1e-3, patience=5,
                        verbose=1, restore_best_weights = True)
model.compile(optimizer='Adam',loss='categorical_crossentropy',metrics=['acc'])
history = model.fit(train_generator,validation_data=test_generator,epochs=100,verbose=2,callbacks=[monitor])
log_loss, acc = model.evaluate(test_generator)
print(log_loss, acc)

Epoch 1/100
8/8 - 16s - 2s/step - acc: 0.6239 - loss: 1.3953 - val_acc: 0.5854 - val_loss: 0.6029
Epoch 2/100
8/8 - 8s - 1s/step - acc: 0.6667 - loss: 0.6076 - val_acc: 0.6829 - val_loss: 0.5790
Epoch 3/100
8/8 - 6s - 805ms/step - acc: 0.7564 - loss: 0.4775 - val_acc: 0.6829 - val_loss: 0.6037
Epoch 4/100
8/8 - 8s - 1s/step - acc: 0.8291 - loss: 0.3737 - val_acc: 0.7317 - val_loss: 0.6629
Epoch 5/100
8/8 - 6s - 775ms/step - acc: 0.8590 - loss: 0.3238 - val_acc: 0.6585 - val_loss: 1.0133
Epoch 6/100
8/8 - 8s - 983ms/step - acc: 0.9060 - loss: 0.2789 - val_acc: 0.6585 - val_loss: 0.9838
Epoch 7/100
8/8 - 7s - 813ms/step - acc: 0.9231 - loss: 0.2195 - val_acc: 0.6098 - val_loss: 0.6635
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 2.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - acc: 0.6740 - loss: 0.5901 
0.5789953470230103 0.6829268336296082


In [4]:
pre_trained = InceptionV3(include_top=False, weights='imagenet',classes=2, input_shape=(150, 150, 3))

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [13]:
pre_trained.trainable = False

In [18]:
tf.random.set_seed(25)
model = tf.keras.Sequential([
    pre_trained,
    tf.keras.layers.Flatten(), # Added Flatten layer here
    tf.keras.layers.Dense(1024, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax') # Changed activation to softmax
])

print(model.summary())

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inception_v3 (Functional)       │ (None, 3, 3, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1024)           │    18,875,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_97          │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,945,186 (156.19 MB)

 Trainable params: 19,140,354 (73.01 MB)

 Non-trainable params: 21,804,832 (83.18 MB)

None


In [ ]:
monitor = EarlyStopping(monitor='val_loss',min_delta=1e-3, patience=5,
                        verbose=1, restore_best_weights = True)
model.compile(optimizer='Adamax',loss='categorical_crossentropy',metrics=['acc']) # Changed loss to categorical_crossentropy
history = model.fit(train_generator,validation_data=test_generator,epochs=100,verbose=2,callbacks=[monitor])
log_loss, acc = model.evaluate(test_generator)
print(log_loss, acc)

Epoch 1/100
